## Phase 3: Learning-to-Rank (LTR) Re-ranking

Two-stage retrieval: FAISS surfaces a broad semantic candidate pool (top 50), then a re-ranker
narrows it down using signals FAISS distance alone can't capture - community rating and
popularity. The re-ranker below uses fixed, hand-picked weights; Phase 5 replaces them with
weights learned from data.

In [14]:
import os
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sentence_transformers import SentenceTransformer
import faiss
import math
import json
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Ridge
import string
import pickle
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from rank_bm25 import BM25Okapi

### Persisting embeddings, the FAISS index, and BM25 tokens

Re-encoding ~20k descriptions, rebuilding the HNSW index, and re-tokenizing for BM25 each
take real time and produce the exact same result every time they're re-run on the same data,
so all three are cached to disk on the first run and reloaded on every subsequent one - each
in its own dedicated, single-purpose file:

- **`data/embeddings.npy`** - the raw embedding matrix (`np.save`/`np.load`), row-aligned
  with `df` by position.
- **`data/faiss_index.bin`** - the FAISS `IndexHNSWFlat`, saved with `faiss.write_index()` /
  reloaded with `faiss.read_index()`.
- **`data/tokens.pkl`** - the tokenized descriptions used to build the BM25 index (see the
  BM25 section below), saved as a pickled list of token lists, row-aligned with `df`.

`df` itself is **not** cached: merging the source CSVs takes a couple of seconds, so it's
always rebuilt fresh from `cleaned_dataset.csv` + `DM1_game_dataset.csv`.

Each cached artifact is also checked against `len(df)` on load (row count must match)
before being trusted.

If you change the embedding model, the source CSVs, or the preprocessing, delete
`embeddings.npy`, `faiss_index.bin`, and/or `tokens.pkl`so they get rebuilt.

### Re-ranking function

Weights (`0.5` semantic / `0.3` rating / `0.2` popularity) are a manual heuristic, not learned -
a reasonable starting point given no labeled training data exists yet at this point in the
notebook. It's the baseline the Phase 5 Ridge model is later compared against.

In [ ]:
DATA_DIR = '../data'
EMBEDDINGS_PATH = os.path.join(DATA_DIR, 'embeddings.npy')
FAISS_INDEX_PATH = os.path.join(DATA_DIR, 'faiss_index.bin')

model = SentenceTransformer('all-MiniLM-L6-v2')

# df is always rebuilt fresh.
df_clean = pd.read_csv(os.path.join(DATA_DIR, 'cleaned_dataset.csv'))
df_raw = pd.read_csv(os.path.join(DATA_DIR, 'DM1_game_dataset.csv'))
df_text = df_raw[['BGGId', 'Description']]
df = pd.merge(df_clean, df_text, on='BGGId', how='inner')
df = df.dropna(subset=['Description']).reset_index(drop=True)
print(f"Dataset ready! Total games: {len(df)}")

# Embeddings: load from cache if present and row-count-valid, else compute 
embeddings = None
if os.path.exists(EMBEDDINGS_PATH):
    cached_embeddings = np.load(EMBEDDINGS_PATH)
    if cached_embeddings.shape[0] == len(df):
        embeddings = cached_embeddings.astype('float32')
        print(f"Embeddings loaded from cache! Shape: {embeddings.shape}")
    else:
        print(
            f"Cache found at '{EMBEDDINGS_PATH}' but row count doesn't match df "
            f"({cached_embeddings.shape[0]} vs {len(df)}) - treating as stale, will recompute."
        )

if embeddings is None:
    print("Encoding descriptions for the LTR engine...")
    embeddings = model.encode(df['Description'].tolist(), show_progress_bar=True)
    embeddings = np.array(embeddings).astype('float32')
    np.save(EMBEDDINGS_PATH, embeddings)
    print(f"Embeddings computed and cached to '{EMBEDDINGS_PATH}'. Shape: {embeddings.shape}")

# FAISS index: load from cache if present and vector-count-valid, else build
dimension = embeddings.shape[1]
index = None
if os.path.exists(FAISS_INDEX_PATH):
    cached_index = faiss.read_index(FAISS_INDEX_PATH)
    if cached_index.ntotal == len(df):
        index = cached_index
        print(f"FAISS index loaded from cache! Total vectors: {index.ntotal}")
    else:
        print(
            f"Cache found at '{FAISS_INDEX_PATH}' but vector count doesn't match df "
            f"({cached_index.ntotal} vs {len(df)}) - treating as stale, will rebuild."
        )

if index is None:
    index = faiss.IndexHNSWFlat(dimension, 32)
    print("Building the HNSW graph...")
    index.add(embeddings)
    faiss.write_index(index, FAISS_INDEX_PATH)
    print(f"FAISS index built and cached to '{FAISS_INDEX_PATH}'. Total vectors: {index.ntotal}")

print("Setup complete! You can now run the LTR search function.")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 16571.28it/s]


Dataset ready! Total games: 19728
Embeddings loaded from cache! Shape: (19728, 384)
FAISS index loaded from cache! Total vectors: 19728
Setup complete! You can now run the LTR search function.


In [16]:
def search_and_rerank_ltr(query, top_candidates=50, final_top_k=5):
    """
    Two-Stage Retrieval: Retrieves candidates via FAISS, then re-ranks them using LTR features.
    """
    # STAGE 1: Fast Candidate Retrieval (FAISS)
    query_vector = model.encode([query]).astype('float32')
    distances, indices = index.search(query_vector, top_candidates)
    
    # Extract the candidate dataframe
    candidates = df.iloc[indices[0]].copy()
    candidates['faiss_distance'] = distances[0]
    
    # STAGE 2: Feature Extraction & Normalization
    scaler = MinMaxScaler()
    
    # 1. Normalize FAISS Distance (Invert it so Higher = Better)
    # We subtract from 1 so a distance of 0 becomes a score of 1.0
    candidates['semantic_score'] = 1 - scaler.fit_transform(candidates[['faiss_distance']])
    
    # 2. Normalize Numerical Metadata (from your original Data Mining project)
    candidates['norm_rating'] = scaler.fit_transform(candidates[['Rating']])
    candidates['norm_popularity'] = scaler.fit_transform(candidates[['Avg_popularity_log']])
    
    # STAGE 3: Pointwise LTR Inference
    # In a fully supervised system, LambdaMART/XGBoost learns these weights.
    # Here, we simulate the inference of a trained pointwise ranker.
    W_SEMANTIC = 0.50    # Text relevance is the most important
    W_RATING = 0.30      # Game quality
    W_POPULARITY = 0.20  # How well-known it is
    
    # Calculate the final ranking score
    candidates['final_ltr_score'] = (
        (candidates['semantic_score'] * W_SEMANTIC) +
        (candidates['norm_rating'] * W_RATING) +
        (candidates['norm_popularity'] * W_POPULARITY)
    )
    
    # Sort by the new LTR score and return the top K
    final_results = candidates.sort_values(by='final_ltr_score', ascending=False).head(final_top_k)
    
    return final_results[['Name', 'final_ltr_score', 'semantic_score', 'norm_rating', 'Rating', 'Description']]

### Test the LTR engine
Same query used throughout notebooks 6-8, so results are directly comparable across BM25,
pure FAISS, and this re-ranked version.

In [17]:
user_query = "A cooperative space game involving betrayal"
print(f"LTR Re-ranked Results for: '{user_query}'\n")

ltr_results = search_and_rerank_ltr(user_query, top_candidates=50, final_top_k=5)

for i, row in ltr_results.iterrows():
    print(f"{row['Name']}")
    print(f"[LTR Score: {row['final_ltr_score']:.2f} | Semantic: {row['semantic_score']:.2f} | Rating: {row['Rating']:.1f}]")
    print(f"Snippet: {row['Description'][:150]}...\n")

LTR Re-ranked Results for: 'A cooperative space game involving betrayal'

The Crew: The Quest for Planet Nine
[LTR Score: 1.00 | Semantic: 1.00 | Rating: 3.0]
Snippet: cooperative tricktaking game crew quest planet player set astronaut uncertain space adventure rumor unknown planet eventful journey space extend   exc...

Cosmic Encounter
[LTR Score: 0.81 | Semantic: 0.62 | Rating: 3.0]
Snippet: build galactic empirein depth space alien race cosmo vie control universe alliance form shift moment moment cataclysmic battle send starship scream wa...

Star Clicker
[LTR Score: 0.64 | Semantic: 0.62 | Rating: 3.0]
Snippet: star clicker new cooperative game christophe raimbault   author colt express   player kid try save planet evil alien parent mission far far awayldquoe...

The Expanse Board Game
[LTR Score: 0.62 | Semantic: 0.38 | Rating: 3.0]
Snippet: expanse board game base syfy television series focus politic conquest intrigue similar board game twilight struggle short playing time cardd

## Phase 4: System Evaluation (Effectiveness Measures)

Three hand-labeled queries, used as a first sanity check on the metric implementations before
scaling up to an LLM-as-a-judge over 25 queries in Phase 4.1 - small enough to eyeball by hand,
big enough to catch an obviously broken MRR/NDCG calculation.

Relevance scale: 3 = perfect match, 2 = highly relevant, 1 = partially relevant, 0 = irrelevant.

In [18]:
# Ground Truth dictionary (Qrels)
qrels = {
    "A cooperative space game involving betrayal": {
        "Battlestar Galactica: The Board Game": 3,
        "Nemesis": 3,
        "Unfathomable": 2, # Betrayal, but not space (it's ocean/Cthulhu)
        "The Crew: The Quest for Planet Nine": 1, # Space and coop, but no betrayal
        "Catan": 0
    },
    "Fast paced card game for two players": {
        "7 Wonders Duel": 3,
        "Jaipur": 3,
        "Lost Cities": 3,
        "Dominion": 2, # Card game, supports 2 players, but maybe not purely "fast paced 2p only"
        "Twilight Imperium": 0 # Exact opposite
    },
    "Heavy economic eurogame with route building": {
        "Brass: Birmingham": 3,
        "Brass: Lancashire": 3,
        "Great Western Trail": 3,
        "Ticket to Ride": 1, # Route building, but very light (not heavy economic)
        "Monopoly": 0
    }
}

test_queries = list(qrels.keys())
print(f"Loaded {len(test_queries)} test queries.")

Loaded 3 test queries.


## Metric Calculation Functions

MRR and NDCG implemented directly rather than pulled from a library, mainly for control over
tie-breaking and to keep the exact rank-discounting logic visible for the report. The two are
complementary: MRR only cares about the position of the *first* relevant hit, NDCG grades the
whole top-k by relevance level - a system can win on one and lose on the other (see the Phase 5
discussion below).

In [19]:
def calculate_mrr(retrieved_games, query_qrels):
    """
    Calculates the Reciprocal Rank for a single query.
    Considers a document 'relevant' if its qrel score is > 0.
    """
    for rank, game_name in enumerate(retrieved_games, start=1):
        # Check if the game is in our ground truth and has a score > 0
        if query_qrels.get(game_name, 0) > 0:
            return 1.0 / rank
    return 0.0

def calculate_ndcg(retrieved_games, query_qrels, k=5):
    """
    Calculates NDCG@K for a single query.
    """
    # 1. Calculate DCG
    dcg = 0.0
    for i in range(min(k, len(retrieved_games))):
        game_name = retrieved_games[i]
        rel_score = query_qrels.get(game_name, 0)
        # Using the standard DCG formula
        dcg += rel_score / math.log2((i + 1) + 1)
        
    # 2. Calculate IDCG (Ideal DCG)
    # Sort the ground truth scores in descending order
    ideal_scores = sorted(list(query_qrels.values()), reverse=True)
    idcg = 0.0
    for i in range(min(k, len(ideal_scores))):
        idcg += ideal_scores[i] / math.log2((i + 1) + 1)
        
    # Prevent division by zero
    if idcg == 0.0:
        return 0.0
        
    # 3. NDCG
    return dcg / idcg

## Model Comparison: BM25 vs Learning-to-Rank

Checks whether the FAISS + heuristic re-ranking pipeline actually beats plain lexical search
(BM25) on this small labeled set, before spending API budget generating a larger evaluation set.

In [20]:
TOKENS_PATH = os.path.join(DATA_DIR, 'tokens.pkl')

# 1. Define the preprocessing function
stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    if not isinstance(text, str):
        return []
    text = text.lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    tokens = word_tokenize(text)
    return [word for word in tokens if word not in stop_words]

# 2. Tokenize the descriptions - reuse the cached tokens if present and valid,
#    otherwise compute once and persist to their own dedicated file.
tokens_list = None
if os.path.exists(TOKENS_PATH):
    with open(TOKENS_PATH, 'rb') as f:
        cached_tokens = pickle.load(f)
    if len(cached_tokens) == len(df):
        tokens_list = cached_tokens
        print("Reusing cached tokens from disk...")
    else:
        print(
            f"Cache found at '{TOKENS_PATH}' but row count doesn't match df "
            f"({len(cached_tokens)} vs {len(df)}) - treating as stale, will recompute."
        )

if tokens_list is None:
    print("Tokenizing descriptions for BM25 (takes a minute)...")
    tokens_list = df['Description'].apply(preprocess_text).tolist()
    with open(TOKENS_PATH, 'wb') as f:
        pickle.dump(tokens_list, f)
    print(f"Tokens computed and cached to '{TOKENS_PATH}'.")

df['tokens'] = tokens_list

# 3. Build the BM25 Index
print("Building BM25 Index...")
bm25 = BM25Okapi(df['tokens'].tolist())

# 4. Define the BM25 search function
def search_games_bm25(query, top_k=5):
    query_tokens = preprocess_text(query)
    doc_scores = bm25.get_scores(query_tokens)

    # We use a copy of the dataframe here to avoid modifying the original one during testing
    df_temp = df.copy()
    df_temp['bm25_score'] = doc_scores
    results = df_temp.sort_values(by='bm25_score', ascending=False).head(top_k)
    return results

print("BM25 is now active in memory! You can run the evaluation cell.")

Reusing cached tokens from disk...
Building BM25 Index...
BM25 is now active in memory! You can run the evaluation cell.


In [21]:
bm25_mrr_total = 0
bm25_ndcg_total = 0
ltr_mrr_total = 0
ltr_ndcg_total = 0

print("Starting Evaluation...\n")

for query in test_queries:
    print(f"Testing Query: '{query}'")
    query_qrels = qrels[query]
    
    # -- 1. Test BM25 --
    # Assuming search_games_bm25 is defined and returns a dataframe
    bm25_results = search_games_bm25(query, top_k=5)['Name'].tolist()
    bm25_mrr = calculate_mrr(bm25_results, query_qrels)
    bm25_ndcg = calculate_ndcg(bm25_results, query_qrels, k=5)
    
    # -- 2. Test LTR --
    ltr_results = search_and_rerank_ltr(query, top_candidates=50, final_top_k=5)['Name'].tolist()
    ltr_mrr = calculate_mrr(ltr_results, query_qrels)
    ltr_ndcg = calculate_ndcg(ltr_results, query_qrels, k=5)
    
    # Accumulate totals
    bm25_mrr_total += bm25_mrr
    bm25_ndcg_total += bm25_ndcg
    ltr_mrr_total += ltr_mrr
    ltr_ndcg_total += ltr_ndcg
    
    print(f"  BM25 -> MRR: {bm25_mrr:.2f} | NDCG@5: {bm25_ndcg:.2f}")
    print(f"  LTR  -> MRR: {ltr_mrr:.2f} | NDCG@5: {ltr_ndcg:.2f}\n")

# Calculate Averages
num_queries = len(test_queries)

print("FINAL AVERAGE SCORES")

print(f"BM25 Baseline:")
print(f"  Mean MRR:    {bm25_mrr_total / num_queries:.3f}")
print(f"  Mean NDCG@5: {bm25_ndcg_total / num_queries:.3f}")
print(f"\nLTR Pipeline:")
print(f"  Mean MRR:    {ltr_mrr_total / num_queries:.3f}")
print(f"  Mean NDCG@5: {ltr_ndcg_total / num_queries:.3f}")

Starting Evaluation...

Testing Query: 'A cooperative space game involving betrayal'
  BM25 -> MRR: 0.00 | NDCG@5: 0.00
  LTR  -> MRR: 1.00 | NDCG@5: 0.16

Testing Query: 'Fast paced card game for two players'
  BM25 -> MRR: 0.00 | NDCG@5: 0.00
  LTR  -> MRR: 0.00 | NDCG@5: 0.00

Testing Query: 'Heavy economic eurogame with route building'
  BM25 -> MRR: 0.00 | NDCG@5: 0.00
  LTR  -> MRR: 0.00 | NDCG@5: 0.00

FINAL AVERAGE SCORES
BM25 Baseline:
  Mean MRR:    0.000
  Mean NDCG@5: 0.000

LTR Pipeline:
  Mean MRR:    0.333
  Mean NDCG@5: 0.053


## Phase 4.1: Automated Qrel Generation (LLM-as-a-Judge)

Hand-labeling qrels doesn't scale past a handful of queries, so the remaining ground truth (25
queries) is generated with an LLM judge (`gemini-3.5-flash-lite`) instead. Candidates are
pooled from the top 10 of BM25 and top 10 of FAISS+re-ranking per query, so the judge only ever
sees candidates either system could plausibly retrieve - grading against documents neither
system has a chance of ranking would be pointless.

**Ranking, not scoring.** The model ranks the whole pool for a query rather than assigning a
raw 0-3 score to each game independently; the score is then derived deterministically from rank
position (top 2 -> 3, next 4 -> 2, next 6 -> 1, rest -> 0). Asking for a raw score directly lets
the model's idea of "3 = perfect match" drift between queries; asking it to rank sidesteps that
entirely. Every query is generated with this same method and model - a checkpoint built
partially with a different method is discarded and regenerated rather than mixed in.

The API call itself - rate-limiting, retries, JSON parsing - lives in `scripts/generate_qrels.py`
and runs as a separate batch job producing `automated_qrels.json`; it doesn't belong inline in a
notebook meant to be re-run interactively.

In [22]:
# Load the pre-generated Qrels from data/evaluation/
with open('../data/evaluation/automated_qrels.json', 'r') as f:
    saved = json.load(f)
full_qrels = saved.get('qrels', saved)

print(f"Successfully loaded {len(full_qrels)} annotated queries!")

Successfully loaded 25 annotated queries!


### Verifying the qrels

Three checks before trusting this as ground truth: judged names match `df['Name']` exactly, the
bucket sizes actually held for every query (no query with more than 2 games scored 3), and a
manual spot-check of a few queries to catch cases where the judge was too generous.

In [23]:
import json
from collections import Counter
from difflib import get_close_matches

known_names = set(df['Name'])
N_SCORE_3 = 2

with open('../data/evaluation/automated_qrels.json', 'r') as f:
    saved = json.load(f)
qrels = saved['qrels']
print(f"Method: {saved['_metadata'].get('method')} | Model: {saved['_metadata'].get('model')}")

print(f"Total queries: {len(qrels)}")  # expected: 25
total_judgments = sum(len(v) for v in qrels.values())
print(f"Total judgments: {total_judgments}\n")

# 1. Exact name match against df['Name'], across all 25 queries
all_ok = True
for query, scores in qrels.items():
    mismatches = [name for name in scores if name not in known_names]
    if mismatches:
        all_ok = False
        print(f"Query: '{query}'")
        for name in mismatches:
            close = get_close_matches(name, known_names, n=2, cutoff=0.6)
            print(f"  [MISMATCH] '{name}' (score {scores[name]}) -> possible: {close}")
        print()
if all_ok:
    print("All judged names match df['Name'] exactly across all 25 queries.\n")

# 2. Label distribution per query - flags any query where the "top 2 -> perfect" bucket
#    wasn't respected (shouldn't happen with the fixed-bucket method, but worth checking)
for query, scores in qrels.items():
    dist = Counter(scores.values())
    n_perfect = dist.get(3, 0)
    flag = " <-- CHECK" if n_perfect > N_SCORE_3 else ""
    print(f"{query[:50]:50s} | 3:{dist.get(3,0):2d} 2:{dist.get(2,0):2d} 1:{dist.get(1,0):2d} 0:{dist.get(0,0):2d}{flag}")


Method: ranking_bucket | Model: gemini-3.5-flash-lite
Total queries: 25
Total judgments: 462

All judged names match df['Name'] exactly across all 25 queries.

Cyberpunk game with hacking mechanics              | 3: 2 2: 4 1: 6 0: 6
Lovecraftian horror game set in the 1920s          | 3: 2 2: 4 1: 6 0: 6
Space exploration and empire building              | 3: 2 2: 4 1: 6 0: 5
Farming simulator with animal breeding             | 3: 2 2: 4 1: 6 0: 6
Medieval trading in the Mediterranean              | 3: 2 2: 4 1: 6 0: 7
Surviving a zombie apocalypse with limited resourc | 3: 2 2: 4 1: 6 0: 6
Pirate adventure game with ship combat             | 3: 2 2: 4 1: 6 0: 3
Deck builder with worker placement                 | 3: 2 2: 4 1: 6 0: 7
Hidden movement game where one player is hunted    | 3: 2 2: 4 1: 6 0: 8
Engine builder with drafting                       | 3: 2 2: 4 1: 6 0: 8
Social deduction game with hidden roles            | 3: 2 2: 4 1: 6 0: 6
Tile placement game with pattern buil

Full listing below, sorted by relevance, for a manual spot-check of the flagged queries (or any
others).

In [24]:
for query, scores in qrels.items():
    print(f"\n{query}")
    for game, score in sorted(scores.items(), key=lambda x: -x[1]):
        print(f"  {score}  {game}")


Cyberpunk game with hacking mechanics
  3  Super Hack Override
  3  Intelle
  2  Black Hat
  2  Bitcoin Hackers
  2  Zaibatsu
  2  Shadowrun: Sprawl Ops
  1  Shadowrun Crossfire: Prime Runner Edition
  1  Shadowrun: Crossfire
  1  Shadowrun: The Trading Card Game
  1  .hack//ENEMY
  1  Aetherium
  1  Human Interface: Be a Better Human
  0  Cyberpunk: The Collectible Card Game
  0  Martian Rails
  0  Sabotage
  0  Covert
  0  Alien Puppies
  0  Resident Evil 2: The Board Game

Lovecraftian horror game set in the 1920s
  3  Arkham Horror
  3  Arkham Horror: The Card Game
  2  Mansions of Madness: Second Edition
  2  Mythos Tales
  2  Miskatonic Madness
  2  Mad Love
  1  Vudulhu
  1  Cthulhu's Vault
  1  Building an Elder God
  1  Shuffle Heroes
  1  Horrified
  1  Final Girl
  0  Claustrophobia 1643
  0  Claustrophobia
  0  Last Friday
  0  Pittsburgh 68
  0  Dungeons & Dragons: Tomb of Annihilation Board Game
  0  Monopoly: Simpsons Treehouse of Horror

Space exploration and empire bu

## Phase 4.2: Large-Scale Evaluation (BM25 vs. LTR)

Same comparison as Phase 4, now over 25 LLM-judged queries instead of 3 hand-labeled ones -
enough to say something about average behavior rather than reading into 3 data points.

In [25]:
# 1. Load the automated Qrels
with open('../data/evaluation/automated_qrels.json', 'r') as f:
    saved = json.load(f)
full_qrels = saved.get('qrels', saved)

# 2. Initialize the metric counters
bm25_mrr_total, bm25_ndcg_total = 0, 0
ltr_mrr_total, ltr_ndcg_total = 0, 0
num_queries = len(full_qrels)

print(f"Evaluating BM25 vs. LTR on {num_queries} queries...\n")

# 3. Evaluation Loop
for query, query_qrels in full_qrels.items():
    
    # --- BM25 Evaluation ---
    bm25_results = search_games_bm25(query, top_k=5)['Name'].tolist()
    bm25_mrr = calculate_mrr(bm25_results, query_qrels)
    bm25_ndcg = calculate_ndcg(bm25_results, query_qrels, k=5)
    
    # --- LTR Evaluation ---
    ltr_results = search_and_rerank_ltr(query, top_candidates=50, final_top_k=5)['Name'].tolist()
    ltr_mrr = calculate_mrr(ltr_results, query_qrels)
    ltr_ndcg = calculate_ndcg(ltr_results, query_qrels, k=5)
    
    # Accumulate the results
    bm25_mrr_total += bm25_mrr
    bm25_ndcg_total += bm25_ndcg
    ltr_mrr_total += ltr_mrr
    ltr_ndcg_total += ltr_ndcg

# 4. Print the final results

print("FINAL AVERAGE RESULTS (25 QUERIES)")

print("BM25 Baseline (Lexical Search):")
print(f"  Mean MRR:    {bm25_mrr_total / num_queries:.3f}")
print(f"  Mean NDCG@5: {bm25_ndcg_total / num_queries:.3f}")

print("\nLTR Pipeline (Semantic + Metadata):")
print(f"  Mean MRR:    {ltr_mrr_total / num_queries:.3f}")
print(f"  Mean NDCG@5: {ltr_ndcg_total / num_queries:.3f}")


Evaluating BM25 vs. LTR on 25 queries...

FINAL AVERAGE RESULTS (25 QUERIES)
BM25 Baseline (Lexical Search):
  Mean MRR:    0.960
  Mean NDCG@5: 0.632

LTR Pipeline (Semantic + Metadata):
  Mean MRR:    0.913
  Mean NDCG@5: 0.486


## Phase 5: Machine-Learned Ranking (Hybrid Search with Linear Regression)

Learns the re-ranking weights instead of hand-picking them, using the 25 LLM-judged queries as
training signal (20 train / 5 held-out test, split by query so no game from a test query leaks
into training). Features: `bm25_score`, `semantic_score`, `norm_rating`, `norm_popularity` -
the same signal types as the Phase 3 heuristic, plus BM25, so this pool blends lexical and
semantic retrieval rather than starting from FAISS candidates alone.

Ridge (`alpha=1.0`) rather than plain linear regression: with only 20 training queries and 4
features, an unregularized fit risks weights that swing on noise; L2 regularization keeps them
stable. `linear_ranker.coef_` prints the actual learned weights for direct comparison against
the fixed `0.5/0.3/0.2` used in Phase 3.

In [26]:
print("1. Building the LTR Training Dataset...")

def extract_query_features(query, query_qrels):
    # Retrieve a broad pool: Top 50 BM25 + Top 50 FAISS
    bm25_res = search_games_bm25(query, top_k=50)
    
    query_vector = model.encode([query]).astype('float32')
    distances, indices = index.search(query_vector, 50)
    faiss_res = df.iloc[indices[0]].copy()
    faiss_res['faiss_distance'] = distances[0]
    
    # Merge results (removing duplicates)
    pool_names = set(bm25_res['Name']).union(set(faiss_res['Name']))
    pool_df = df[df['Name'].isin(pool_names)].copy()
    
    # Recalculate BM25 for the entire pool
    query_tokens = preprocess_text(query)
    pool_df['bm25_raw'] = bm25.get_scores(query_tokens)[pool_df.index]
    
    # Recalculate FAISS for the entire pool
    pool_embeddings = np.array(model.encode(pool_df['Description'].tolist())).astype('float32')
    faiss_distances = [np.linalg.norm(query_vector - emb) for emb in pool_embeddings]
    pool_df['faiss_raw'] = faiss_distances
    
    # --- FEATURE NORMALIZATION ---
    scaler = MinMaxScaler()
    
    # BM25: Higher is better
    pool_df['bm25_score'] = scaler.fit_transform(pool_df[['bm25_raw']])
    
    # FAISS: Lower is better (distance), so we invert it (1 - normalized distance)
    pool_df['semantic_score'] = 1 - scaler.fit_transform(pool_df[['faiss_raw']])
    
    # Global metadata
    pool_df['norm_rating'] = scaler.fit_transform(pool_df[['Rating']])
    pool_df['norm_popularity'] = scaler.fit_transform(pool_df[['Avg_popularity_log']])
    
    # --- TARGET (GROUND TRUTH) ---
    pool_df['relevance_label'] = pool_df['Name'].apply(lambda x: query_qrels.get(x, 0))
    pool_df['query'] = query
    
    return pool_df[['query', 'Name', 'bm25_score', 'semantic_score', 'norm_rating', 'norm_popularity', 'relevance_label']]

# Assemble the dataset for all 25 queries
all_data = []
for query, qrels in full_qrels.items():
    query_df = extract_query_features(query, qrels)
    all_data.append(query_df)

ltr_dataset = pd.concat(all_data, ignore_index=True)
print(f"Dataset created: {len(ltr_dataset)} (Query, Game) pairs extracted.")

# 2. Train/Test Split based on QUERIES (Not individual rows!)
queries = list(full_qrels.keys())
train_queries, test_queries = train_test_split(queries, test_size=5, random_state=42)

train_data = ltr_dataset[ltr_dataset['query'].isin(train_queries)]
test_data = ltr_dataset[ltr_dataset['query'].isin(test_queries)]

features = ['bm25_score', 'semantic_score', 'norm_rating', 'norm_popularity']
X_train = train_data[features]
y_train = train_data['relevance_label']

# 3. Train the Linear Model (Ridge)
print("\n2. Training Ridge Regression (Pointwise LTR)...")
linear_ranker = Ridge(alpha=1.0)
linear_ranker.fit(X_train, y_train)

# These are the actual "weights" learned by the model!
print("\nWEIGHTS LEARNED BY THE MODEL")
for feature, weight in zip(features, linear_ranker.coef_):
    print(f"  {feature}: {weight:.3f}")

# 4. Evaluation on the Test Set (The 5 unseen queries)
print("\n3. Evaluation on the Test Set (5 Secret Queries)...")
mlr_mrr_total, mlr_ndcg_total = 0, 0
bm25_test_mrr, bm25_test_ndcg = 0, 0

for query in test_queries:
    query_qrels = full_qrels[query]
    query_test_data = test_data[test_data['query'] == query].copy()
    
    # BM25 Baseline
    bm25_results = query_test_data.sort_values('bm25_score', ascending=False).head(5)['Name'].tolist()
    bm25_test_mrr += calculate_mrr(bm25_results, query_qrels)
    bm25_test_ndcg += calculate_ndcg(bm25_results, query_qrels, k=5)
    
    # LTR Model (Linear Regression)
    query_test_data['pred_score'] = linear_ranker.predict(query_test_data[features])
    mlr_results = query_test_data.sort_values('pred_score', ascending=False).head(5)['Name'].tolist()
    mlr_mrr_total += calculate_mrr(mlr_results, query_qrels)
    mlr_ndcg_total += calculate_ndcg(mlr_results, query_qrels, k=5)

num_test = len(test_queries)

print("TEST SET RESULTS (5 QUERIES)")
print(f"BM25 Baseline:")
print(f"  Mean MRR:    {bm25_test_mrr / num_test:.3f}")
print(f"  Mean NDCG@5: {bm25_test_ndcg / num_test:.3f}")
print(f"\nLinear MLR (Hybrid Search):")
print(f"  Mean MRR:    {mlr_mrr_total / num_test:.3f}")
print(f"  Mean NDCG@5: {mlr_ndcg_total / num_test:.3f}")

1. Building the LTR Training Dataset...
Dataset created: 2357 (Query, Game) pairs extracted.

2. Training Ridge Regression (Pointwise LTR)...

WEIGHTS LEARNED BY THE MODEL
  bm25_score: 0.806
  semantic_score: 0.638
  norm_rating: 0.238
  norm_popularity: 0.287

3. Evaluation on the Test Set (5 Secret Queries)...
TEST SET RESULTS (5 QUERIES)
BM25 Baseline:
  Mean MRR:    0.900
  Mean NDCG@5: 0.575

Linear MLR (Hybrid Search):
  Mean MRR:    1.000
  Mean NDCG@5: 0.535


**MRR 1.000 vs 0.900**: the hybrid model puts a relevant game in position #1 for every held-out
query; BM25 misses on one. **NDCG@5 0.535 vs 0.575**: BM25 is still slightly better at ordering
positions #2-5. Net read: the learned weights are very good at picking the single best answer,
but don't yet beat lexical matching at fine-grained ordering further down the list - plausible
given only 25 training queries to fit on.